In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/DatasetAnnotation.pdf
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/ECCV16_Paper_Mueller.pdf
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/ReadMe.txt
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/configSeqs.m
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123/bike1.txt
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123/bike2.txt
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123/bike3.txt
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123/bird1_1.txt
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123/bird1_2.txt
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123/bird1_3.txt
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123/boat1.txt
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123/boat2.txt
/kaggle/input/d

In [2]:
!ls -la /kaggle/input/
!ls -la /kaggle/input/notebooks/*/* 2>/dev/null || true
!ls -la /kaggle/input/datasets/*/* 2>/dev/null || true

total 16
drwxr-xr-x 4 root root 4096 Nov 19 06:59 .
drwxr-xr-x 8 root root 4096 Nov 19 06:59 ..
drwxr-xr-x 3 root root 4096 Nov 19 06:59 datasets
drwxr-xr-x 3 root root 4096 Nov 19 06:59 notebooks
total 31238
-rw-rw-rw- 1 root root        0 Nov 19 06:36 custom.css
-rw-rw-rw- 1 root root 12209553 Nov 19 06:36 __notebook__.ipynb
-rw-rw-rw- 1 root root  8510362 Nov 19 06:36 __output__.json
drwxrwxrwx 1 root root        0 Nov 19 06:59 proj
-rw-rw-rw- 1 root root 11266733 Nov 19 06:36 __results__.html
total 8247
drwxrwxrwx 1 root root       0 Nov 19 06:59 anno
-rw-rw-rw- 1 root root   20594 Nov 18 18:24 configSeqs.m
drwxrwxrwx 1 root root       0 Nov 19 06:59 data_seq
-rw-rw-rw- 1 root root  419429 Nov 18 18:24 DatasetAnnotation.pdf
-rw-rw-rw- 1 root root 8000782 Nov 18 18:24 ECCV16_Paper_Mueller.pdf
-rw-rw-rw- 1 root root    2457 Nov 18 18:24 ReadMe.txt


In [3]:
%%bash
PROJ=$(ls -d /kaggle/input/notebooks/*/*/proj)
echo "PROJ=$PROJ"
ls "$PROJ/tools"
find "$PROJ" -maxdepth 3 -name "*eval*.py"
python "$PROJ/tools/eval.py" --help 2>&1 || true

PROJ=/kaggle/input/notebooks/aayushipriyam/notebook404434d840/proj
demo.py
eval.py
test.py
train.py
/kaggle/input/notebooks/aayushipriyam/notebook404434d840/proj/tools/eval.py
Traceback (most recent call last):
  File "/kaggle/input/notebooks/aayushipriyam/notebook404434d840/proj/tools/eval.py", line 11, in <module>
    from toolkit.datasets import UAV10Dataset,UAV20Dataset,DTBDataset,UAVDataset
ModuleNotFoundError: No module named 'toolkit'


In [4]:
import glob, os
import numpy as np

PROJ = "/kaggle/input/notebooks/aayushipriyam/notebook404434d840/proj"
RES_DIR = f"{PROJ}/results/UAV10fps"
ANNO_DIR = "/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123"

def load_boxes(path):
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.replace(",", " ").split()
            rows.append([float(x) for x in parts[:4]])
    return np.array(rows, dtype=np.float64)

def iou(a, b):
    x1 = np.maximum(a[:, 0], b[:, 0])
    y1 = np.maximum(a[:, 1], b[:, 1])
    x2 = np.minimum(a[:, 0] + a[:, 2], b[:, 0] + b[:, 2])
    y2 = np.minimum(a[:, 1] + a[:, 3], b[:, 1] + b[:, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    union = a[:, 2] * a[:, 3] + b[:, 2] * b[:, 3] - inter
    return np.where(union > 0, inter / union, 0.0)

thresholds = np.linspace(0, 1, 21)
results = []

for tracker in sorted(os.listdir(RES_DIR)):
    t_dir = os.path.join(RES_DIR, tracker)
    if not os.path.isdir(t_dir):
        continue
    aucs, precs = [], []
    for pred_file in sorted(glob.glob(os.path.join(t_dir, "*.txt"))):
        vid = os.path.basename(pred_file)
        gt_file = os.path.join(ANNO_DIR, vid)
        if not os.path.exists(gt_file):
            continue
        pred, gt = load_boxes(pred_file), load_boxes(gt_file)

        # Subsample 30fps UAV123 ground truth to 10fps
        if len(gt) > len(pred) * 2:
            gt = gt[::3]

        n = min(len(pred), len(gt))
        pred, gt = pred[:n], gt[:n]

        valid = ~np.isnan(gt).any(axis=1)
        pred, gt = pred[valid], gt[valid]

        overlaps = iou(pred, gt)
        aucs.append(np.mean([(overlaps > t).mean() for t in thresholds]))

        c_pred = pred[:, :2] + pred[:, 2:] / 2.0
        c_gt = gt[:, :2] + gt[:, 2:] / 2.0
        dist = np.linalg.norm(c_pred - c_gt, axis=1)
        precs.append((dist <= 20.0).mean())

    results.append((tracker, np.mean(aucs), np.mean(precs), len(aucs)))

print(f"{'Tracker':<18} {'Success (AUC)':>14} {'Precision (@20px)':>18} {'Videos':>8}")
print("-" * 62)
for name, auc, prec, n_vid in sorted(results, key=lambda x: x[1], reverse=True):
    print(f"{name:<18} {auc:>14.4f} {prec:>18.4f} {n_vid:>8}")

Tracker             Success (AUC)  Precision (@20px)   Videos
--------------------------------------------------------------
interp_auto1.2             0.5776             0.7361      123
base1.2                    0.5759             0.7343      123
z0901.2                    0.3255             0.4810      123
z1151.2                    0.2336             0.4056      123
z1301.2                    0.0708             0.2121      123
z0801.2                    0.0675             0.1634      123
z1501.2                    0.0510             0.1882      123


In [5]:
!python -c "import torch; assert torch.cuda.is_available(), 'GPU not enabled!'; print('GPU:', torch.cuda.get_device_name(0))"
!rm -rf /kaggle/working/proj
!git clone -b main https://github.com/vision4robotics/HiFT.git /kaggle/working/proj
%cd /kaggle/working/proj
!pip -q install yacs tqdm colorama pyyaml matplotlib opencv-python gdown

GPU: Tesla T4
Cloning into '/kaggle/working/proj'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 376 (delta 0), reused 1 (delta 0), pack-reused 373 (from 1)
Receiving objects: 100% (376/376), 45.39 MiB | 51.24 MiB/s, done.
Resolving deltas: 100% (141/141), done.
/kaggle/working/proj


In [6]:
import os
os.chdir('/kaggle/working')

In [7]:
%%bash
set -e
cd /kaggle/working
python -c "import torch; assert torch.cuda.is_available(), 'GPU not enabled!'; print('GPU:', torch.cuda.get_device_name(0))"
rm -rf /kaggle/working/proj
git clone -b main https://github.com/vision4robotics/HiFT.git /kaggle/working/proj
pip -q install yacs tqdm colorama pyyaml matplotlib opencv-python gdown
ls -la /kaggle/working/proj

GPU: Tesla T4
total 64
drwxr-xr-x 10 root root  4096 Nov 19 07:01 .
drwxr-xr-x  3 root root  4096 Nov 19 07:01 ..
drwxr-xr-x  2 root root  4096 Nov 19 07:01 experiments
drwxr-xr-x  8 root root  4096 Nov 19 07:01 .git
drwxr-xr-x  2 root root  4096 Nov 19 07:01 image
-rw-r--r--  1 root root 11357 Nov 19 07:01 LICENSE
drwxr-xr-x  2 root root  4096 Nov 19 07:01 pretrained_models
drwxr-xr-x  8 root root  4096 Nov 19 07:01 pysot
-rw-r--r--  1 root root  4599 Nov 19 07:01 README.md
-rw-r--r--  1 root root    77 Nov 19 07:01 requirement.txt
drwxr-xr-x  7 root root  4096 Nov 19 07:01 toolkit
drwxr-xr-x  2 root root  4096 Nov 19 07:01 tools
drwxr-xr-x  6 root root  4096 Nov 19 07:01 training_dataset


Cloning into '/kaggle/working/proj'...


In [8]:
import os, pathlib

os.chdir('/kaggle/working/proj')
os.makedirs('snapshot', exist_ok=True)
!gdown --fuzzy "https://drive.google.com/file/d/1X8vScXvZ1QohbqzE9EAheQ_yaJCTslSA/view?usp=sharing" -O snapshot/first.pth

def patch(path, old, new):
    p = pathlib.Path(path)
    s = p.read_text()
    if new in s:
        return
    assert s.count(old) >= 1, f"{path}: target snippet not found"
    p.write_text(s.replace(old, new))

# 1. config.py
cfgp = pathlib.Path('pysot/core/config.py')
if 'TRACK.ZOOM' not in cfgp.read_text():
    with open(cfgp, 'a') as f:
        f.write('''
__C.TRACK.ZOOM = 1.0
__C.TRACK.ZOOM_MODE = 'fixed'
__C.TRACK.ZOOM_THR = 40.0
__C.TRACK.INTERP = 'linear'
''')

# 2. hift_tracker.py
tp = 'pysot/tracker/hift_tracker.py'
patch(tp, "    def init(self, img, bbox):", '''    def _pick_zoom(self, size):
        k = float(cfg.TRACK.ZOOM)
        if cfg.TRACK.ZOOM_MODE == 'small' and np.sqrt(size[0] * size[1]) >= cfg.TRACK.ZOOM_THR:
            return 1.0
        return k

    def init(self, img, bbox):''')

patch(tp, "        s_z = round(np.sqrt(w_z * h_z))\n",
          "        self.zoom = self._pick_zoom(self.size)\n"
          "        s_z = round(np.sqrt(w_z * h_z) / self.zoom)\n")

patch(tp, "        s_z = np.sqrt(w_z * h_z)\n",
          "        s_z = np.sqrt(w_z * h_z) / self.zoom\n")

# 3. base_tracker.py
bp = 'pysot/tracker/base_tracker.py'
patch(bp, "class SiameseTracker(BaseTracker):", '''_INTERP = {
    'linear': cv2.INTER_LINEAR, 'cubic': cv2.INTER_CUBIC,
    'lanczos': cv2.INTER_LANCZOS4, 'area': cv2.INTER_AREA
}

def _pick_interp(src, dst):
    m = cfg.TRACK.INTERP
    if m == 'auto':
        return cv2.INTER_CUBIC if src < dst else cv2.INTER_AREA
    return _INTERP[m]

class SiameseTracker(BaseTracker):''')

patch(bp, "im_patch = cv2.resize(im_patch, (model_sz, model_sz))",
          "im_patch = cv2.resize(im_patch, (model_sz, model_sz),\n"
          "                                  interpolation=_pick_interp(im_patch.shape[0], model_sz))")

# 4. model_load.py (PyTorch 2.6+ compatibility — replaces both occurrences)
patch('pysot/utils/model_load.py',
      "map_location=lambda storage, loc: storage.cuda(device))",
      "map_location=lambda storage, loc: storage.cuda(device), weights_only=False)")

print("✅ Checkpoint downloaded and all patches applied.")

Downloading...
From (original): https://drive.google.com/uc?id=1X8vScXvZ1QohbqzE9EAheQ_yaJCTslSA
From (redirected): https://drive.google.com/uc?id=1X8vScXvZ1QohbqzE9EAheQ_yaJCTslSA&confirm=t&uuid=68d5b4e8-2b5a-41bd-97c3-c3c14a807374
To: /kaggle/working/proj/snapshot/first.pth
100%|██████████████████████████████████████| 86.0M/86.0M [00:02<00:00, 41.9MB/s]
✅ Checkpoint downloaded and all patches applied.


In [9]:
import os, sys, shutil

os.chdir('/kaggle/working/proj')
if '.' not in sys.path:
    sys.path.insert(0, '.')

SRC = '/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset'
DST = '/tmp/test_dataset'
os.makedirs(DST, exist_ok=True)

uav20_dir = f'{DST}/UAV123_20L'
if os.path.exists(uav20_dir):
    shutil.rmtree(uav20_dir)
os.makedirs(f'{uav20_dir}/data_seq', exist_ok=True)
os.symlink(f'{SRC}/anno/UAV20L', f'{uav20_dir}/anno')

for txt in sorted(os.listdir(f'{SRC}/anno/UAV20L')):
    if txt.endswith('.txt'):
        seq_name = txt[:-4]
        src_seq = f'{SRC}/data_seq/UAV123/{seq_name}'
        if os.path.isdir(src_seq):
            os.symlink(src_seq, f'{uav20_dir}/data_seq/{seq_name}')

os.system(f'ln -sfn {uav20_dir}/data_seq {DST}/UAV20l')
os.system(f'ln -sfn {DST} /kaggle/working/proj/test_dataset')

from toolkit.datasets import DatasetFactory
ds = DatasetFactory.create_dataset(name='UAV20l', dataset_root='./test_dataset/UAV20l', load_img=False)
print(f"✅ UAV20l ready: loaded {len(ds)} videos")

loading UAV20l: 100%|█████████████████████████████████████████| 20/20 [00:02<00:00,  8.77it/s, uav1]

✅ UAV20l ready: loaded 20 videos


In [10]:
import os, yaml, copy

os.chdir('/kaggle/working/proj')
base = yaml.safe_load(open('experiments/config.yaml'))

def make_cfg(name, **trk):
    c = copy.deepcopy(base)
    c.setdefault('TRACK', {}).update(trk)
    path = f'experiments/{name}.yaml'
    yaml.safe_dump(c, open(path, 'w'))
    return path

runs = {
    'base': {},
    'z090': {'ZOOM': 0.9},
    'z130': {'ZOOM': 1.3},
    'z150': {'ZOOM': 1.5},
    'interp_auto': {'INTERP': 'auto'}
}

DS = 'UAV20l'
for n, kw in runs.items():
    cfg_path = make_cfg(n, **kw)
    snap_link = f'snapshot/{n}.pth'
    if not os.path.exists(snap_link):
        os.symlink('/kaggle/working/proj/snapshot/first.pth', snap_link)
    print(f"\n=== Running {n} on {DS} ===")
    !PYTHONPATH=. python tools/test.py --dataset {DS} --snapshot {snap_link} --config {cfg_path}


=== Running base on UAV20l ===
loading UAV20l: 100%|████████████████████████████████████████| 20/20 [00:00<00:00, 143.00it/s, uav1]
(  1) Video: bike1        Time:  68.4s Speed: 45.1fps
(  2) Video: bird1        Time:  86.5s Speed: 28.2fps
(  3) Video: car1         Time:  82.7s Speed: 31.8fps
(  4) Video: car16        Time:  42.1s Speed: 47.3fps
(  5) Video: car3         Time:  34.8s Speed: 49.3fps
(  6) Video: car6         Time: 249.0s Speed: 19.5fps
(  7) Video: car8         Time:  54.6s Speed: 47.1fps
(  8) Video: car9         Time:  38.2s Speed: 49.2fps
(  9) Video: group1       Time: 100.2s Speed: 48.6fps
( 10) Video: group2       Time:  64.5s Speed: 41.6fps
( 11) Video: group3       Time: 147.7s Speed: 37.4fps
( 12) Video: person14     Time:  63.9s Speed: 45.7fps
( 13) Video: person17     Time:  63.0s Speed: 37.3fps
( 14) Video: person19     Time:  96.5s Speed: 45.1fps
( 15) Video: person2      Time:  56.5s Speed: 46.4fps
( 16) Video: person20     Time:  47.9s Speed: 37.2fps
( 1

In [11]:
import os, glob
import numpy as np

RES_DIR = '/kaggle/working/proj/results/UAV20l'
ANNO_DIR = '/tmp/test_dataset/UAV123_20L/anno'

gt_map = {}
for p in sorted(glob.glob(f'{ANNO_DIR}/*.txt')):
    vid = os.path.splitext(os.path.basename(p))[0]
    gt_map[vid] = np.loadtxt(p, delimiter=',', dtype=np.float64)

def eval_sequence(pred, gt):
    n = min(len(pred), len(gt))
    pred, gt = pred[:n], gt[:n]
    valid = (~np.isnan(gt).any(axis=1)) & (gt[:, 2] > 0) & (gt[:, 3] > 0)

    iou = np.zeros(n, dtype=np.float64)
    if valid.any():
        px1, py1 = pred[valid, 0], pred[valid, 1]
        px2, py2 = px1 + pred[valid, 2], py1 + pred[valid, 3]
        gx1, gy1 = gt[valid, 0], gt[valid, 1]
        gx2, gy2 = gx1 + gt[valid, 2], gy1 + gt[valid, 3]
        inter = np.maximum(0.0, np.minimum(px2, gx2) - np.maximum(px1, gx1)) * \
                np.maximum(0.0, np.minimum(py2, gy2) - np.maximum(py1, gy1))
        union = (pred[valid, 2] * pred[valid, 3]) + (gt[valid, 2] * gt[valid, 3]) - inter
        iou[valid] = np.where(union > 0, inter / union, 0.0)

    cle = np.full(n, np.inf, dtype=np.float64)
    if valid.any():
        pcx, pcy = pred[valid, 0] + pred[valid, 2] / 2.0, pred[valid, 1] + pred[valid, 3] / 2.0
        gcx, gcy = gt[valid, 0] + gt[valid, 2] / 2.0, gt[valid, 1] + gt[valid, 3] / 2.0
        cle[valid] = np.hypot(pcx - gcx, pcy - gcy)

    auc = np.mean([(iou > t).mean() for t in np.linspace(0.0, 1.0, 21)])
    prec20 = (cle <= 20.0).mean()
    return auc, prec20

rows = []
for tracker in sorted(os.listdir(RES_DIR)):
    t_dir = os.path.join(RES_DIR, tracker)
    if not os.path.isdir(t_dir):
        continue
    aucs, precs = [], []
    for txt in sorted(glob.glob(f'{t_dir}/*.txt')):
        vid = os.path.splitext(os.path.basename(txt))[0]
        if vid in gt_map:
            a, p = eval_sequence(np.loadtxt(txt, delimiter=','), gt_map[vid])
            aucs.append(a)
            precs.append(p)
    if aucs:
        rows.append((tracker, np.mean(aucs), np.mean(precs), len(aucs)))

rows.sort(key=lambda x: x[1], reverse=True)
print(f"{'Tracker':<20} {'Success (AUC)':>14} {'Precision (@20px)':>18} {'Videos':>8}")
print("-" * 64)
for name, auc, prec, nv in rows:
    print(f"{name:<20} {auc:>14.4f} {prec:>18.4f} {nv:>8d}")

Tracker               Success (AUC)  Precision (@20px)   Videos
----------------------------------------------------------------
base1.2                      0.5280             0.6727       20
interp_auto1.2               0.5125             0.6560       20
z0901.2                      0.2383             0.3769       20
z1501.2                      0.0221             0.1302       20
z1301.2                      0.0161             0.0868       20
